# 🎯 [OPS02_M10] 다중 센서 중복 항적 통합 시뮬레이터 최종 빌드 및 통합 방공망 브리핑 패키지 완성
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS02/ops02_m10_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS02_M10 |
| **미션 주제** | 다중 센서 중복 항적 통합 시뮬레이터 최종 빌드 및 통합 방공망 브리핑 패키지 완성 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 간 무력 충돌 사례에서는 드론·미사일이 여러 방향에서 동시에 들어오면서 레이더, 광학(EO/IR), 음향 센서가 같은 비행체를 각자 경보해 상황판에 중복 항적이 쏟아지고, 이 때문에 위협 수를 잘못 세거나 대응이 늦어지는 문제가 주목받았다. 본 프로젝트에서는 가상 방어 지역에 배치한 가상 센서 4기와 가상 비행체 궤적으로 교육용 경보 데이터를 코드로 만들어, 연동 규격과 통합 스키마 설계, 시각 동기화와 ENU 좌표 변환 전처리, 최단거리 매칭 Baseline, 확장 칼만 필터(EKF) 항적 추적·통합, 융합 신뢰도 지수, 전술 상황판(COP), 조작관 HITL 검토, 통신 단절·재밍 스트레스 시험, AAR 재보정, 최종 통합까지 다중 센서 중복 항적 통합 시뮬레이터를 단계별로 구현한다. 최종 목표는 비행체 하나를 하나의 항적으로 정확히 보여 주어 지휘관의 인지 부하를 줄이고, 항적 통합·분리 같은 판단은 조작관이 근거와 함께 확정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/ops02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 승인 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/ops02_data_spec.md`, `docs/ops02_hitl_guide.md`, `docs/ops02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/ops02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/ops02_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/ops02_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 다중 센서 중복 항적 통합 시뮬레이터 최종 빌드 및 통합 방공망 브리핑 패키지 완성
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

다중 센서 중복 항적 통합 시뮬레이터 전체 모듈을 통합 검증하고 합동 방공 작전 운용을 위한 브리핑 패키지와 배포 가이드를 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **메시지 검증부터 시공간 전처리, 항적 융합, 신뢰도 등급, 조작관 검토 대기열, 감사 로그까지 한 번에 실행되는 통합 시뮬레이터와 방공망 브리핑을 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **손상 메시지를 차단하고 재보정 파라미터로 융합한 뒤, Med 항적을 전량 조작관 검토 대기열에 올린다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/main_ops02.py, README.md, docs/ops02_final_report.pdf(md)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무 즉시 도입 가능성, 운용 신뢰도 및 전략적 기대효과

#### 미션 PASS 조건

- 정량 기준: End-to-End 통합 파이프라인 실행 성공률 100%, 테스트 커버리지 85% 이상


- 진행 기준: 원클릭 실행 스크립트 동작 확인 및 최종 발표 산출물 작성 완료


- 검증 방법: 최종 검증 위원회(평가관) 모의 시연 및 전체 기능 통합 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""센서 메시지 검증·정제, 시각 동기화·ENU 변환, 재보정 파라미터를 적용한 EKF 항적 융합, 융합 신뢰도 등급, 조작관 검토 대기열·감사 로그까지 한 번에 실행하는 통합 시뮬레이터를 만들고, 자체 점검(함수 커버리지 포함)과 통합 방공망 브리핑 요약을 출력한다."""
import hashlib
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# ---------------------------------------------------------------------------
# [교육용가상자료] 다중 센서(레이더·EO/IR·음향) 경보 생성기 (실제 장비 제원·배치와 무관)
# 가상 방어 지역(원점 기준 동/북 km)에 비행체 궤적을 만들고, 센서마다 '자기 위치 기준
# 거리·방위'와 '자기 시계 기준 시각'으로 경보를 낸다. 같은 비행체가 여러 센서에 동시에
# 잡히므로 경보가 중복되고, 센서마다 시계 오차·측정 오차·오경보가 다르다.
# ---------------------------------------------------------------------------
SENSORS = {
    "RDR-A": {"type": "radar", "pos": (0.0, 0.0), "sr": 0.08, "saz": 0.4, "pd": 0.95, "fa": 1.0, "max_r": 40.0, "clock": 0.0},
    "RDR-B": {"type": "radar", "pos": (16.0, -6.0), "sr": 0.10, "saz": 0.5, "pd": 0.93, "fa": 1.0, "max_r": 40.0, "clock": 0.42},
    "EOIR-C": {"type": "eoir", "pos": (5.0, 9.0), "sr": 0.40, "saz": 0.15, "pd": 0.88, "fa": 0.5, "max_r": 18.0, "clock": -0.65},
    "ACU-D": {"type": "acoustic", "pos": (10.0, 2.0), "sr": 0.60, "saz": 2.0, "pd": 0.80, "fa": 0.6, "max_r": 9.0, "clock": 0.93},
}
CENTER = np.array([8.0, 2.0])                         # 가상 방어 지역 중심
KINDS = {"uav": (0.03, 0.06), "heli": (0.04, 0.07), "fast": (0.15, 0.20)}   # 속도 범위(km/s, 추상값)
WEATHER = {"clear": {"eoir": 1.0, "acoustic": 1.0, "radar": 1.0, "fa": 1.0},
           "night": {"eoir": 0.9, "acoustic": 1.0, "radar": 1.0, "fa": 1.1},
           "fog": {"eoir": 0.55, "acoustic": 1.0, "radar": 1.0, "fa": 1.2},
           "rain": {"eoir": 0.7, "acoustic": 0.6, "radar": 0.97, "fa": 1.6}}


def _targets(rng, n, T, special):
    out, kinds = {}, {}
    for i in range(n):
        kind = "fast" if special == "fast" and i < 2 else str(rng.choice(["uav", "uav", "heli", "fast"]))
        ang = rng.uniform(0, 2 * np.pi)
        p = CENTER + rng.uniform(6, 12) * np.array([np.sin(ang), np.cos(ang)])
        hd = np.arctan2(*(CENTER - p)) + rng.normal(0, 0.35)          # 방어 지역 쪽으로(북 기준 방위)
        sp = rng.uniform(*KINDS[kind])
        omega = rng.uniform(0.04, 0.08) * rng.choice([-1, 1]) if (special == "turn" and i < 3) or rng.random() < 0.25 else 0.0
        if special == "formation" and i == 0:
            omega = 0.0
        if special == "cross" and i < 2:                              # 두 표적이 중심에서 교차
            p = CENTER + np.array([-6.0, 0.0]) if i == 0 else CENTER + np.array([0.0, -6.0])
            hd, sp, omega, kind = (np.pi / 2 if i == 0 else 0.0), 0.1, 0.0, "heli"
        if special == "formation" and i == 1:                         # 0.4 km 간격 편대
            p0 = out["TGT00"][0]
            p, hd, sp, omega, kind = p0[:2] + np.array([0.4, 0.0]), np.arctan2(p0[2], p0[3]), np.hypot(p0[2], p0[3]), 0.0, kinds["TGT00"]
        traj = np.zeros((T, 4))
        for t in range(T):
            if t > 0:
                if 20 <= t < 40:
                    hd += omega                                       # 20~40초 선회 기동
                p = p + sp * np.array([np.sin(hd), np.cos(hd)])
            traj[t] = [p[0], p[1], sp * np.sin(hd), sp * np.cos(hd)]
        out[f"TGT{i:02d}"], kinds[f"TGT{i:02d}"] = traj, kind
    return out, kinds


def make_scenario(seed, n=6, T=60, weather="clear", fa_mult=1.0, dropout=None, ghost=False, special=None,
                  noise_mult=None, drift=0.0):
    """시나리오 1개: 정답 궤적, 센서 경보(보고) 목록, 시계 동기화용 기준 신호(heartbeat)를 만든다.
    dropout=(센서, 시작, 끝): 통신 단절, ghost: 한 센서에만 보이는 거짓 항적(기만·스푸핑),
    noise_mult={센서: 배수}: 재밍으로 측정 잡음 증가, drift: 음향 센서 시계가 초당 drift초씩 흐트러짐."""
    rng = np.random.default_rng(seed)
    targets, kinds = _targets(rng, n, T, special)
    w = WEATHER[weather]
    reports, hb, mid = [], [], 0
    gpos = CENTER + np.array([-9.0, 7.0])
    for t in range(T):
        for sid, s in SENSORS.items():
            clock = s["clock"] + (drift * t if sid == "ACU-D" else 0.0)
            if t % 10 == 0:
                hb.append({"sensor": sid, "t_ref": float(t), "t_sensor": t + clock + rng.normal(0, 0.02)})
            if dropout and sid == dropout[0] and dropout[1] <= t < dropout[2]:
                continue
            nm = (noise_mult or {}).get(sid, 1.0)
            objs = [(tid, tr[t, :2]) for tid, tr in targets.items()]
            if ghost and sid == "EOIR-C":
                objs.append(("GHOST", gpos + 0.12 * t * np.array([0.6, -0.8])))
            for tid, xy in objs:
                d = xy - np.array(s["pos"])
                r = float(np.hypot(*d))
                if r > s["max_r"] or rng.random() > s["pd"] * w[s["type"]]:
                    continue
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(r + rng.normal(0, s["sr"] * nm), 3),
                                "az_deg": round(float((np.degrees(np.arctan2(d[0], d[1])) + rng.normal(0, s["saz"] * nm)) % 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": tid})
                mid += 1
            for _ in range(rng.poisson(s["fa"] * fa_mult * w["fa"] * (3 if nm > 1 else 1))):   # 오경보(클러터·반사)
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(rng.uniform(0.5, s["max_r"]), 3), "az_deg": round(rng.uniform(0, 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": "FA"})
                mid += 1
    return {"targets": targets, "kinds": kinds, "reports": reports, "heartbeats": hb, "T": T, "weather": weather}
# ---------------------------------------------------------------------------

# ---- 시공간 전처리 공통 함수: 정제 -> 시각 동기화 -> 공통 좌표(ENU, km) 변환 ----
def estimate_offsets(heartbeats):
    """센서별 시계 오차 = 기준 신호(heartbeat) 대비 센서 시각 차이의 중앙값(잡음에 강함)."""
    df = pd.DataFrame(heartbeats)
    return (df["t_sensor"] - df["t_ref"]).groupby(df["sensor"]).median().to_dict()


def clean(reports):
    """중복 메시지·결측·범위 밖(반사파 등 최대 탐지거리 초과) 경보를 걸러낸다. (정제 목록, 사유별 건수)"""
    seen, out, why = set(), [], {"중복": 0, "결측": 0, "범위 밖": 0}
    for r in reports:
        if r["msg_id"] in seen:
            why["중복"] += 1; continue
        seen.add(r["msg_id"])
        if any(r.get(k) is None or (isinstance(r.get(k), float) and np.isnan(r[k])) for k in ("t_sensor", "range_km", "az_deg")):
            why["결측"] += 1; continue
        if not 0 <= r["range_km"] <= SENSORS[r["sensor"]]["max_r"] * 1.1:
            why["범위 밖"] += 1; continue
        out.append(r)
    return out, why


def to_enu(r, offsets):
    """센서 국소 극좌표(거리·방위) -> 공통 ENU(동·북 km), 센서 시각 -> 기준 시각."""
    s, a = SENSORS[r["sensor"]], np.radians(r["az_deg"])
    return {**r, "t": r["t_sensor"] - offsets.get(r["sensor"], 0.0),
            "x": s["pos"][0] + r["range_km"] * np.sin(a), "y": s["pos"][1] + r["range_km"] * np.cos(a)}


def meas_cov(r):
    """거리·방위 오차를 ENU 공분산(2x2)으로 변환한다(1차 근사)."""
    a, rr = np.radians(r["az_deg"]), r["range_km"]
    J = np.array([[np.sin(a), rr * np.cos(a)], [np.cos(a), -rr * np.sin(a)]])
    return J @ np.diag([r["sr"] ** 2, np.radians(r["saz"]) ** 2]) @ J.T


def prepare(scn, sync=True):
    reps, _ = clean(scn["reports"])
    off = estimate_offsets(scn["heartbeats"]) if sync else {}
    return sorted((to_enu(r, off) for r in reps), key=lambda r: r["t"])

# ---- 확장 칼만 필터(EKF) 다중 센서 항적 추적기 공통 함수 ----
# 상태 x = [동(km), 북(km), 동속(km/s), 북속(km/s)], 측정 z = 센서 기준 [거리(km), 방위(rad)]
def ekf_predict(x, P, dt, q):
    F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    G = np.array([[dt ** 2 / 2, 0], [0, dt ** 2 / 2], [dt, 0], [0, dt]])
    return F @ x, F @ P @ F.T + q * G @ G.T


def h_polar(x, spos):
    dx, dy = x[0] - spos[0], x[1] - spos[1]
    r2 = max(dx * dx + dy * dy, 1e-9); r = np.sqrt(r2)
    return np.array([r, np.arctan2(dx, dy)]), np.array([[dx / r, dy / r, 0, 0], [dy / r2, -dx / r2, 0, 0]])


def innovation(x, P, rep):
    s = SENSORS[rep["sensor"]]
    zp, H = h_polar(x, s["pos"])
    y = np.array([rep["range_km"], np.radians(rep["az_deg"])]) - zp
    y[1] = (y[1] + np.pi) % (2 * np.pi) - np.pi                      # 방위 차이는 -pi~pi로 감싼다
    S = H @ P @ H.T + np.diag([rep["sr"] ** 2, np.radians(rep["saz"]) ** 2])
    return y, H, S, float(y @ np.linalg.solve(S, y))


class Tracker:
    """센서 경보를 스캔(1초) 단위로 받아 EKF 항적을 갱신한다. 같은 스캔 안에서 센서별로
    순차 갱신하므로, 여러 센서가 같은 비행체를 보면 하나의 항적으로 통합된다."""

    def __init__(self, gate=9.21, q=4e-4, confirm=(3, 5), max_miss=4, merge_gate=6.0):
        self.gate, self.q, self.confirm, self.max_miss, self.merge_gate = gate, q, confirm, max_miss, merge_gate
        self.tracks, self.dead, self.nid, self.n_merged = [], [], 0, 0

    def _new(self, rep, k):
        C = meas_cov(rep)
        P = np.zeros((4, 4)); P[:2, :2] = C; P[2, 2] = P[3, 3] = 0.25 ** 2
        self.tracks.append({"id": f"FT{self.nid:03d}", "x": np.array([rep["x"], rep["y"], 0, 0]), "P": P, "k": k,
                            "scans": [k], "sensors": {rep["sensor"]: 1}, "truths": [rep["truth"]], "nis": [],
                            "hist": [(k, rep["x"], rep["y"])], "confirmed": False, "miss": 0, "born": k, "upd": True})
        self.nid += 1

    def _assign(self, tracks, rs, sid):
        """게이트(마할라노비스 거리) 안에서 전역 최근접 할당(GNN). 할당된 보고서 번호 집합을 돌려준다."""
        if not tracks or not rs:
            return set()
        spos = SENSORS[sid]["pos"]
        Z = np.array([[r["range_km"], np.radians(r["az_deg"])] for r in rs])
        R = np.diag([rs[0]["sr"] ** 2, np.radians(rs[0]["saz"]) ** 2])
        cost = np.full((len(tracks), len(rs)), 1e6)
        for i, tr in enumerate(tracks):
            zp, H = h_polar(tr["x"], spos)
            Y = Z - zp
            Y[:, 1] = (Y[:, 1] + np.pi) % (2 * np.pi) - np.pi
            Si = np.linalg.inv(H @ tr["P"] @ H.T + R)
            d2 = np.einsum("ij,jk,ik->i", Y, Si, Y)
            cost[i] = np.where(d2 < self.gate, d2, 1e6)
        used = set()
        for i, j in zip(*linear_sum_assignment(cost)):
            if cost[i, j] < self.gate:
                tr, r = tracks[i], rs[j]
                y, H, S, d2 = innovation(tr["x"], tr["P"], r)
                K = tr["P"] @ H.T @ np.linalg.inv(S)
                tr["x"] = tr["x"] + K @ y
                tr["P"] = (np.eye(4) - K @ H) @ tr["P"]
                tr["nis"].append(d2); tr["truths"].append(r["truth"])
                tr["sensors"][sid] = tr["sensors"].get(sid, 0) + 1
                tr["upd"] = True; used.add(j)
        return used

    def _merge(self):
        """같은 비행체를 따라가는 중복 항적(위치·속도가 통계적으로 같음)을 오래된 항적으로 통합한다."""
        drop = set()
        for a in range(len(self.tracks)):
            for b in range(a + 1, len(self.tracks)):
                A, B = self.tracks[a], self.tracks[b]
                if A["id"] in drop or B["id"] in drop:
                    continue
                d = A["x"] - B["x"]
                d2 = float(d[:2] @ np.linalg.solve(A["P"][:2, :2] + B["P"][:2, :2], d[:2]))
                if d2 < self.merge_gate and np.hypot(d[2], d[3]) < 0.05:
                    keep, lose = (A, B) if len(A["scans"]) >= len(B["scans"]) else (B, A)
                    for sid, n in lose["sensors"].items():
                        keep["sensors"][sid] = keep["sensors"].get(sid, 0) + n
                    keep["truths"] += lose["truths"]; keep["scans"] = sorted(set(keep["scans"]) | set(lose["scans"]))
                    keep["merged"] = keep.get("merged", 0) + 1
                    drop.add(lose["id"])
        self.tracks = [t for t in self.tracks if t["id"] not in drop]
        self.n_merged += len(drop)

    def step(self, k, reps):
        for tr in self.tracks:                                         # 1) 예측
            tr["x"], tr["P"] = ekf_predict(tr["x"], tr["P"], k - tr["k"], self.q)
            tr["k"], tr["upd"] = k, False
        for sid in sorted({r["sensor"] for r in reps}, key=lambda s: SENSORS[s]["sr"]):   # 2) 정밀 센서부터 순차 갱신
            rs = [r for r in reps if r["sensor"] == sid]
            used = self._assign([t for t in self.tracks if t["confirmed"]], rs, sid)      # 확정 항적 우선
            rest = [r for j, r in enumerate(rs) if j not in used]
            used2 = self._assign([t for t in self.tracks if not t["confirmed"]], rest, sid)
            for j, r in enumerate(rest):
                if j not in used2:
                    self._new(r, k)
        if self.merge_gate > 0:
            self._merge()
        keep = []
        for tr in self.tracks:                                         # 3) 확정·삭제 관리
            if tr["upd"]:
                tr["miss"] = 0
                if tr["scans"][-1] != k:
                    tr["scans"].append(k)
                tr["hist"].append((k, tr["x"][0], tr["x"][1]))
            else:
                tr["miss"] += 1
            m, n = self.confirm
            if not tr["confirmed"] and sum(1 for s in tr["scans"] if s > k - n) >= m:
                tr["confirmed"] = True
            limit = self.max_miss if tr["confirmed"] else 1
            (keep if tr["miss"] <= limit else self.dead).append(tr)
        self.tracks = keep

    def all_tracks(self):
        return self.dead + self.tracks


def run_tracker(reps, **kw):
    trk = Tracker(**kw)
    by = {}
    for r in reps:
        by.setdefault(int(round(r["t"])), []).append(r)
    for k in range(min(by), max(by) + 1):
        trk.step(k, by.get(k, []))
    return trk.all_tracks()


def label_track(tr):
    vals, cnt = np.unique(tr["truths"], return_counts=True)
    return str(vals[np.argmax(cnt)]), float(cnt.max() / cnt.sum())


def evaluate(tracks, scn, min_reports=5):
    """정답 연결 평가: 재현율, 항적 분할(한 표적에 항적 2개 이상), 거짓 항적, 병합 오류, 위치 오차."""
    conf = [t for t in tracks if t["confirmed"]]
    for t in conf:
        t["label"], t["purity"] = label_track(t)
    cnt = pd.Series([r["truth"] for r in scn["reports"] if r["truth"].startswith("TGT")]).value_counts()
    truths = [k for k, v in cnt.items() if v >= min_reports]
    per = {g: [t for t in conf if t["label"] == g] for g in truths}
    found = [g for g in truths if per[g]]
    split = sum(max(0, len(v) - 1) for v in per.values())
    false = [t for t in conf if not t["label"].startswith("TGT")]
    merge = [t for t in conf if t["label"].startswith("TGT") and t["purity"] < 0.7]
    err = []
    for g in found:
        for t in per[g]:
            for k, x, y in t["hist"][3:]:
                if 0 <= k < scn["T"]:
                    err.append(np.hypot(x - scn["targets"][g][k, 0], y - scn["targets"][g][k, 1]))
    n = len(truths)
    acc = len(found) / max(n + split + len(false) + len(merge), 1)
    return {"정답표적": n, "탐지표적": len(found), "재현율": round(len(found) / max(n, 1), 3), "분할": split,
            "거짓항적": len(false), "병합오류": len(merge), "위치RMSE(km)": round(float(np.sqrt(np.mean(np.square(err)))) if err else np.nan, 3),
            "항적정확도": round(acc, 3)}

CONFIG = {"tracker": dict(gate=9.21, q=5e-5, confirm=(4, 6), max_miss=1, merge_gate=0.0),        # 미션 9 재보정값
          "high": {"min_sensors": 2, "min_rps": 1.5}}
CALLED = set()


def traced(fn):
    def w(*a, **k):
        CALLED.add(fn.__name__); return fn(*a, **k)
    w.__name__ = fn.__name__
    return w


@traced
def validate(r):
    errs = []
    if r.get("sensor") not in SENSORS:
        errs.append("미등록 센서")
    for k in ("t_sensor", "range_km", "az_deg"):
        if not isinstance(r.get(k), (int, float)) or not np.isfinite(r[k]):
            errs.append(f"{k} 결측·형식"); break
    return errs


@traced
def ingest(scn):
    ok, bad = [], {}
    for r in scn["reports"]:
        e = validate(r)
        (bad.__setitem__(r["msg_id"], e) if e else ok.append(r))
    reps, why = clean(ok)
    off = estimate_offsets(scn["heartbeats"])
    return sorted((to_enu(r, off) for r in reps), key=lambda r: r["t"]), bad, why, off


@traced
def fuse(reps):
    return run_tracker(reps, **CONFIG["tracker"])


@traced
def grade(t):
    n = len(t["scans"])
    hi = len(t["sensors"]) >= CONFIG["high"]["min_sensors"] and len(t["truths"]) / n >= CONFIG["high"]["min_rps"]
    return "High" if hi else "Med"


class Queue:
    def __init__(self):
        self.items, self.log = [], []

    @traced
    def submit(self, rec):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"event": "SUBMIT", "id": rec["id"], "grade": rec["grade"], "reason": rec["reason"]}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()})
        self.items.append({**rec, "status": "PENDING"})                      # 자동 통합·분리·교전 연계 없음

    @traced
    def verify(self):
        prev = "GENESIS"
        for r in self.log:
            body = {k: v for k, v in r.items() if k not in ("prev", "hash")}
            if r["prev"] != prev or r["hash"] != hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest():
                return False
            prev = r["hash"]
        return True


@traced
def main(scn):
    t0 = time.perf_counter()
    reps, bad, why, off = ingest(scn)
    trk = fuse(reps)
    ev = evaluate(trk, scn)
    q, table = Queue(), []
    for t in [t for t in trk if t["confirmed"]]:
        g = grade(t)
        rec = {"id": t["id"], "grade": g, "sensors": "+".join(sorted(t["sensors"])), "scans": len(t["scans"]), "label": t["label"]}
        table.append(rec)
        if g == "Med":
            q.submit({**rec, "reason": "단일 센서·보고 부족(기만·클러터 가능성)"})
    ms = (time.perf_counter() - t0) * 1000 / scn["T"]
    return pd.DataFrame(table), ev, bad, why, off, q, ms, len(reps)


T0 = time.perf_counter()
# [교육용가상자료] 야간, 표적 8개, 기만 항적, RDR-B 20초 단절, 손상 메시지 4건
scn = make_scenario(1501, n=8, T=60, weather="night", ghost=True, dropout=("RDR-B", 25, 45))
for k, (f, v) in enumerate([("range_km", None), ("sensor", "RDR-Z"), ("az_deg", float("nan")), ("t_sensor", "12:00")]):
    bad = dict(scn["reports"][50 + k]); bad[f] = v; bad["msg_id"] = f"BAD{k}"; scn["reports"].append(bad)
D, ev, bad, why, off, q, ms, n_used = main(scn)
elapsed = time.perf_counter() - T0
true_hi = D[(D["grade"] == "High")]["label"].str.startswith("TGT")

chk = {
    "손상 메시지 4건 차단": len(bad) == 4,
    "메시지 건수 보존(유실 0)": n_used + len(bad) + sum(why.values()) == len(scn["reports"]),
    "시계 오차 추정 4개 센서": len(off) == 4,
    "표적 재현율 90% 이상": ev["재현율"] >= 0.9,
    "High 항적 중 거짓 0건": bool(true_hi.all()),
    "Med 항적 전량 검토 대기": len(q.items) == (D["grade"] == "Med").sum(),
    "자동 적용 0건": all(i["status"] == "PENDING" for i in q.items),
    "감사 로그 무결성": q.verify() and len(q.log) == len(q.items),
    "스캔당 200ms 이내": ms < 200,
    "항적 분할 0건": ev["분할"] == 0,
}
passed = sum(chk.values())
for k, v in chk.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
funcs = ["validate", "ingest", "fuse", "grade", "submit", "verify", "main"]
cov = len(CALLED & set(funcs)) / len(funcs)
print(f"자체 점검 {passed}/{len(chk)} 통과, E2E 실행 성공률 {'100%' if passed == len(chk) else f'{passed / len(chk):.0%}'}, 함수 커버리지 {cov:.0%} (목표 85% 이상: {'달성' if cov >= 0.85 else '미달'})")
print("차단 메시지:", bad)
print("\n" + D.drop(columns="label").to_string(index=False))

n_raw = len(scn["reports"])
print(f"""
[통합 방공망 브리핑 요약 - 교육용가상자료]
1. 처리: 센서 4종 경보 {n_raw}건 수신, 손상 {len(bad)}건 차단, 정제 후 {n_used}건 융합, 전체 {elapsed:.1f}s, 스캔당 {ms:.1f} ms
2. 융합: 원시 경보 {n_raw}건 -> 통합 항적 {len(D)}개 (High {(D['grade'] == 'High').sum()} / Med {(D['grade'] == 'Med').sum()}), 표적 재현율 {ev['재현율']:.0%}, 분할 {ev['분할']}건, 위치 오차 {ev['위치RMSE(km)']} km
3. 검토: Med 항적 {len(q.items)}건 조작관 검토 대기(기만·클러터 가능성), 자동 통합·분리 0건, 감사 로그 무결성 {'정상' if q.verify() else '이상'}
4. 운용: 레이더B 단절 구간도 다른 센서로 추적을 유지했다. 항적 결정은 조작관이, 교전 여부는 교전수칙·지휘계통이 정한다.
5. 한계·확장: 가상 센서·가상 좌표 기준 결과로 실제 센서 제원·배치에 맞춘 재보정이 필요하다. 인접 체계 연동은 통합 항적(JSON) 표준 메시지 API로 확장한다.""")

plt.figure(figsize=(6, 6))
for tid, tr in scn["targets"].items():
    plt.plot(tr[:, 0], tr[:, 1], "k-", lw=0.7)
for t in fuse(ingest(scn)[0]):
    if t["confirmed"]:
        h = np.array(t["hist"]); plt.plot(h[:, 1], h[:, 2], "--", c="tab:blue" if grade(t) == "High" else "tab:orange")
for sid, s in SENSORS.items():
    plt.plot(*s["pos"], "^", c="tab:green")
plt.xlabel("East (km)"); plt.ylabel("North (km)"); plt.title("Integrated fusion: High (blue) / Med review (orange) - synthetic"); plt.axis("equal")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[PASS] 손상 메시지 4건 차단
[PASS] 메시지 건수 보존(유실 0)
[PASS] 시계 오차 추정 4개 센서
[PASS] 표적 재현율 90% 이상
[PASS] High 항적 중 거짓 0건
[PASS] Med 항적 전량 검토 대기
[PASS] 자동 적용 0건
[PASS] 감사 로그 무결성
[PASS] 스캔당 200ms 이내
[PASS] 항적 분할 0건
자체 점검 10/10 통과, E2E 실행 성공률 100%, 함수 커버리지 100% (목표 85% 이상: 달성)
차단 메시지: {'BAD0': ['range_km 결측·형식'], 'BAD1': ['미등록 센서'], 'BAD2': ['az_deg 결측·형식'], 'BAD3': ['t_sensor 결측·형식']}

   id grade                  sensors  scans
FT012   Med                   EOIR-C     26
FT000  High ACU-D+EOIR-C+RDR-A+RDR-B     60
FT001  High ACU-D+EOIR-C+RDR-A+RDR-B     60
FT002  High ACU-D+EOIR-C+RDR-A+RDR-B     60
FT003  High ACU-D+EOIR-C+RDR-A+RDR-B     60
FT005  High ACU-D+EOIR-C+RDR-A+RDR-B     60
FT006  High ACU-D+EOIR-C+RDR-A+RDR-B     59
FT007  High ACU-D+EOIR-C+RDR-A+RDR-B     60
FT008  High ACU-D+EOIR-C+RDR-A+RDR-B     60
FT116   Med                   EOIR-C     21

[통합 방공망 브리핑 요약 - 교육용가상자료]
1. 처리: 센서 4종 경보 1655건 수신, 손상 4건 차단, 정제 후 1650건 융합, 전체 0.3s, 스캔당 4.6 ms
2. 융합: 원시 경보 1655건 -> 통합 항적 10개 (High 8 / Med 2), 표적 재현율 100%, 분할 0건, 위치 오차 0.044 km
3. 검토: Med 항적 2건 조작관 검토 대기(기만·클러터 가능성), 자동 통합·분리 0건, 감사 로그 무결성 정상
4. 운용: 레이더B 단절 구간도 다른 센서로 추적을 유지했다. 항적 결정은 조작관이, 교전 여부는 교전수칙·지휘계통이 정한다.
5. 한계·확장: 가상 센서·가상 좌표 기준 결과로 실제 센서 제원·배치에 맞춘 재보정이 필요하다. 인접 체계 연동은 통합 항적(JSON) 표준 메시지 API로 확장한다.
```
</details>
